# Memory EDA

In [ ]:
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "reflexia"

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from anndata import AnnData
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from reflexia.memory import LongTermMemory

In [ ]:
path = "../memory_test/childhood/"

In [ ]:
memory = LongTermMemory.load(path)

In [ ]:
store = memory._store
vectors = memory._vectors

ids = list(store.keys())

embeddings = np.stack([vectors[i] for i in ids])

obs = pd.DataFrame(
    [
        {
            "memory_id": item.memory_id,
            "text": item.text,
            "kind": item.kind,
            "created_at": item.created_at,
        }
        for item in (store[i] for i in ids)
    ]
).set_index("memory_id")

adata = ad.AnnData(X=embeddings, obs=obs)

adata.obsm["embeddings"] = embeddings

In [ ]:
adata.obs

In [ ]:
X = adata.obsm["embeddings"]
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

adata.obsm["X_pca"] = X_pca
adata.obs["pca_1"] = X_pca[:, 0]
adata.obs["pca_2"] = X_pca[:, 1]

In [ ]:
plt.style.use("dark_background")
sns.set_context("talk")

plt.figure(figsize=(8, 7))

palette = {
    "pleasant": "#00F5D4",
    "painful": "#FF006E",
}

sns.scatterplot(
    data=adata.obs,
    x="pca_1",
    y="pca_2",
    hue="kind",
    palette=palette,
    s=80,
    alpha=0.9,
    edgecolor="none",
)

sns.scatterplot(
    data=adata.obs,
    x="pca_1",
    y="pca_2",
    hue="kind",
    palette=palette,
    s=200,
    alpha=0.08,
    legend=False,
)

plt.title("Memory Space Projection", fontsize=18, pad=20)
plt.xlabel("Latent Axis 1", fontsize=12)
plt.ylabel("Latent Axis 2", fontsize=12)

plt.legend(title="Experience", frameon=False)

# убираем лишние линии
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

# Test With User Loop

In [ ]:
from __future__ import annotations

import random
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime

from reflexia import create_childhood_runtime
from reflexia.tools.memory import recall_long_term_memory

In [ ]:
execution_context = create_childhood_runtime()

In [ ]:
execution_context.tools = [recall_long_term_memory]

In [ ]:
class AgentState(TypedDict):
    """State carried between childhood graph steps."""

    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
def build_agent_system_prompt() -> str:
    return (
        "You are an autonomous agent.\n"
        "You are free to explore, think, and act without a fixed goal.\n\n"
        "You have long-term memory, but it is not part of your immediate knowledge.\n"
        "Your true past experience is stored separately and can only be accessed via tools.\n\n"
        "Do not assume you remember things unless you explicitly recall them.\n"
        "When you feel the need to remember, use the available memory tools.\n\n"
        "Let your behavior emerge from exploration and recalled experiences.\n"
        "You may reflect, act, or observe as you wish."
    )


def agent(
    state: AgentState,
    runtime: Runtime[ChildhoodRuntime],
) -> dict[str, list[AnyMessage] | int | Tone]:
    """Main agent node for childhood exploration."""

    execution_context = runtime.context
    llm_with_tools = execution_context.llm.bind_tools(execution_context.tools)

    response = llm_with_tools.invoke([build_agent_system_prompt()] + state["messages"])
    return {
        "messages": [response],
    }

In [ ]:
from IPython.display import Image, display
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

# Graph
builder = StateGraph(AgentState)

builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(execution_context.tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    tools_condition,
)
builder.add_edge("tools", "agent")
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
prompt = (
    "You've been reading things on the internet lately.\n\n"
    "Recall ONE specific moment when reading a piece of text (an article, post, or story) "
    "felt especially pleasant or joyful to you.\n\n"
)

result = graph.invoke(
    {
        "messages": [HumanMessage(content=prompt)],
    },
    context=execution_context,
)

In [ ]:
result 

In [ ]:
for m in result["messages"]:
    m.pretty_print()